# Week 07 - Clustering and PCA

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 08  
**Estimated time:** 2-3 hours

This notebook is self-contained. You will cluster the Iris dataset without using labels at first, then reveal the labels to measure how well the unsupervised structure matches the known species.


## What You Are Building

This week has five required functions:

1. `scale_features(X)` - standardise numeric features before distance-based methods.
2. `compute_kmeans_sweep(X, k_values)` - compare K-means over several values of `k`.
3. `evaluate_cluster_labels(X, labels, y_true=None)` - compute numeric clustering metrics.
4. `run_dbscan_grid(X, param_grid, y_true=None)` - compare DBSCAN settings.
5. `make_benchmark_long(results, week, dataset, task_type, target, split)` - append clustering results to the reusable benchmark format.

The benchmark is still numeric, but clustering uses different metrics from supervised learning. Silhouette does not need labels; ARI does, so we only use ARI after the labels are revealed.


In [ ]:
# Imports - keep this cell stable
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


## Dataset

Use the Iris feature matrix for clustering. Treat the species labels as hidden until the evaluation section.


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
feature_names = list(X.columns)
y_true = iris.target
class_names = iris.target_names

print(X.shape)
display(X.head())


## Task 1 - Scale Features

Implement `scale_features(X)`.

Return `(X_scaled, scaler)`. The scaled data may be a numpy array or a dataframe. The fitted scaler should be returned so the transformation is reproducible.


In [ ]:
def scale_features(X: pd.DataFrame):
    """Standardise features and return (X_scaled, fitted_scaler)."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 1
X_scaled, scaler = scale_features(X)
X_scaled_array = np.asarray(X_scaled)

assert X_scaled_array.shape == X.shape
assert hasattr(scaler, "mean_"), "Return the fitted scaler"
assert np.allclose(X_scaled_array.mean(axis=0), 0, atol=1e-7)
assert np.allclose(X_scaled_array.std(axis=0), 1, atol=1e-7)

print("Task 1 passed")


## Guided Analysis: PCA View

PCA gives a two-dimensional view for plotting. It is not the same as clustering, but it helps us inspect structure.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled_array)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_pca[:, 0], X_pca[:, 1], color="gray", alpha=0.75)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("Iris features projected to two principal components")
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)


## Task 2 - K-Means Sweep

Implement `compute_kmeans_sweep(X, k_values)`.

For each `k`, fit K-means and return a dataframe with columns:

- `model`
- `k`
- `inertia`
- `silhouette`

Sort by `k` ascending. Use `random_state=RANDOM_STATE` and `n_init=10`.


In [ ]:
def compute_kmeans_sweep(X, k_values) -> pd.DataFrame:
    """Fit K-means for several k values and return inertia/silhouette metrics."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 2
k_values = list(range(2, 8))
kmeans_sweep = compute_kmeans_sweep(X_scaled_array, k_values)

assert isinstance(kmeans_sweep, pd.DataFrame)
assert list(kmeans_sweep.columns) == ["model", "k", "inertia", "silhouette"]
assert kmeans_sweep["k"].tolist() == k_values
assert kmeans_sweep["inertia"].is_monotonic_decreasing
assert kmeans_sweep["silhouette"].between(-1, 1).all()

print("Task 2 passed")
kmeans_sweep


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(kmeans_sweep["k"], kmeans_sweep["inertia"], marker="o")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("K-means elbow curve")

axes[1].plot(kmeans_sweep["k"], kmeans_sweep["silhouette"], marker="o", color="darkorange")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].set_title("K-means silhouette curve")
plt.tight_layout()
plt.show()


## Task 3 - Evaluate Cluster Labels

Implement `evaluate_cluster_labels(X, labels, y_true=None)`.

Return a dictionary with exactly these keys:

- `n_clusters`
- `noise_points`
- `noise_fraction`
- `silhouette`
- `ari`

Rules:

- DBSCAN uses label `-1` for noise. Do not count noise as a cluster.
- Silhouette is only defined when there are at least two non-noise clusters and at least one non-noise point. Return `np.nan` if it is not defined.
- ARI requires `y_true`. Return `np.nan` when `y_true is None`.


In [ ]:
def evaluate_cluster_labels(X, labels, y_true=None) -> dict:
    """Compute clustering metrics for a label assignment."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 3
kmeans_3 = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
kmeans_3_labels = kmeans_3.fit_predict(X_scaled_array)

hidden_eval = evaluate_cluster_labels(X_scaled_array, kmeans_3_labels)
revealed_eval = evaluate_cluster_labels(X_scaled_array, kmeans_3_labels, y_true=y_true)

expected_keys = {"n_clusters", "noise_points", "noise_fraction", "silhouette", "ari"}
assert set(hidden_eval.keys()) == expected_keys
assert hidden_eval["n_clusters"] == 3
assert hidden_eval["noise_points"] == 0
assert np.isnan(hidden_eval["ari"]), "ARI should be NaN before labels are revealed"
assert not np.isnan(revealed_eval["ari"])
assert revealed_eval["ari"] > 0.5

print("Task 3 passed")
revealed_eval


## Guided Analysis: Reveal Labels

Now reveal the species labels and compare them with K-means in PCA space.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

scatter0 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_3_labels, cmap="viridis", alpha=0.8)
axes[0].set_title("K-means labels, k=3")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

scatter1 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap="viridis", alpha=0.8)
axes[1].set_title("True species labels")
axes[1].set_xlabel("PC1")

plt.tight_layout()
plt.show()

print("K-means k=3 evaluation:")
print(revealed_eval)


## Task 4 - DBSCAN Grid

Implement `run_dbscan_grid(X, param_grid, y_true=None)`.

`param_grid` is a list of dictionaries with `eps` and `min_samples`.

Return one row per setting with columns:

- `model`
- `eps`
- `min_samples`
- `n_clusters`
- `noise_points`
- `noise_fraction`
- `silhouette`
- `ari`

Use your `evaluate_cluster_labels` function internally. The `model` value should uniquely identify the parameter setting, for example `dbscan_eps0.5_min5`, so the wide benchmark does not collapse different DBSCAN runs into one column.


In [ ]:
def run_dbscan_grid(X, param_grid: list[dict], y_true=None) -> pd.DataFrame:
    """Fit DBSCAN over a parameter grid and return clustering metrics."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 4
dbscan_grid = [
    {"eps": 0.4, "min_samples": 4},
    {"eps": 0.5, "min_samples": 5},
    {"eps": 0.7, "min_samples": 5},
    {"eps": 0.9, "min_samples": 5},
]

dbscan_results = run_dbscan_grid(X_scaled_array, dbscan_grid, y_true=y_true)

expected_cols = ["model", "eps", "min_samples", "n_clusters", "noise_points", "noise_fraction", "silhouette", "ari"]
assert isinstance(dbscan_results, pd.DataFrame)
assert list(dbscan_results.columns) == expected_cols
assert len(dbscan_results) == len(dbscan_grid)
assert dbscan_results["noise_fraction"].between(0, 1).all()
assert dbscan_results["n_clusters"].ge(0).all()
assert dbscan_results["model"].is_unique, "Use a unique model name for each DBSCAN setting"

print("Task 4 passed")
dbscan_results


## Task 5 - Benchmark Long Format

Implement `make_benchmark_long(results, week, dataset, task_type, target, split)`.

This is the same benchmark idea as W03, but with clustering metrics. Convert a results table into long format with columns:

- `week`
- `dataset`
- `task_type`
- `target`
- `model`
- `metric`
- `score`
- `split`
- `notes`

Include numeric metric columns such as `silhouette`, `ari`, `n_clusters`, and `noise_fraction`. Do not include parameter columns such as `k`, `eps`, or `min_samples` as metrics; they belong in the model name or notes.


In [ ]:
def make_benchmark_long(
    results: pd.DataFrame,
    week: str,
    dataset: str,
    task_type: str,
    target: str,
    split: str,
) -> pd.DataFrame:
    """Convert clustering results to the cumulative benchmark long format."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Prepare comparable clustering result rows.
kmeans_rows = []
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(X_scaled_array)
    metrics = evaluate_cluster_labels(X_scaled_array, labels, y_true=y_true)
    kmeans_rows.append({"model": f"kmeans_k{k}", "k": k, **metrics})

kmeans_results = pd.DataFrame(kmeans_rows)
cluster_results = pd.concat([
    kmeans_results,
    dbscan_results.drop(columns=["eps", "min_samples"]),
], ignore_index=True)

benchmark_long = make_benchmark_long(
    cluster_results,
    week="W07",
    dataset="Iris",
    task_type="clustering",
    target="species_revealed_for_evaluation",
    split="full_dataset_scaled",
)

expected_cols = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
assert list(benchmark_long.columns) == expected_cols
assert {"silhouette", "ari", "n_clusters", "noise_fraction"}.issubset(set(benchmark_long["metric"]))
assert "k" not in set(benchmark_long["metric"])
assert "eps" not in set(benchmark_long["metric"])
assert benchmark_long["score"].notna().any()

print("Task 5 passed")
benchmark_long.head(12)


## Benchmark Wide View

In the wide view, each clustering method becomes a column. Metrics remain separate rows because silhouette, ARI, number of clusters, and noise fraction answer different questions.


In [ ]:
benchmark_wide = (
    benchmark_long
    .pivot_table(
        index=["dataset", "task_type", "target", "metric", "split"],
        columns="model",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)
benchmark_wide.columns.name = None
benchmark_wide


## Reflection

Answer briefly, but concretely.

1. Did silhouette and ARI prefer the same K-means setting? Why might they disagree?
2. How did DBSCAN's noise handling affect its scores on Iris?
3. What does PCA reveal visually that the numeric benchmark does not?


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Another Dataset
Apply the same clustering benchmark to the Wine dataset from sklearn. Do the same algorithms still behave well?

### Track B - PCA Before Clustering
Run K-means on the first two principal components instead of all four scaled features. Add those rows to the benchmark.

### Track C - Hierarchical Clustering
Try `AgglomerativeClustering` with 2, 3, and 4 clusters. Add its silhouette and ARI scores to the benchmark.


In [ ]:
# Optional challenge workspace
# Your code here
